# MBAI 448 | Week 8 Assignment: Agentic AI

##### Assignment Overview

This assignment explores how agentic AI can be applied to a real-world problem in customer support. It is organized into three Acts:

- Act I: Understand the problem and context
- Act II: Prototype a solution with AI technology
- Act III: Socialize the work with stakeholders

##### Assignment Tools

This assignment assumes you will be working with GitHub Copilot in VS Code, and will require you to submit your chat history along with this notebook. If you are curious about how to work effectively with GitHub Copilot, please consult the [VS Code documentation](https://code.visualstudio.com/docs/copilot/overview).

Submissions that demonstrate thoughtless interaction with Copilot (e.g., asking Copilot to just read the notebook and produce all the outputs) will receive reduced credit.

You will also need access to a language model that supports tool calling. The `github-copilot-sdk` package is the most convenient option and consistent with your existing Copilot setup. You may alternatively use the Anthropic or OpenAI API directly if you prefer.

## Business Goal / Case Statement

You are a first-level support analyst on the Operations team at **MediaVault**, a regional media distribution company that manages content inventory across multiple store locations. MediaVault handles a catalog of films across genres, manages customer accounts and rental transactions, and employs staff across its locations.

Your inbox receives a steady stream of customer emails — billing questions, complaints, product inquiries, account requests, general feedback. Currently, you handle each one manually: read the email, look up the customer in the system, investigate the issue, check company policies, and either respond directly or escalate to a specialist. It's repetitive, time-consuming, and inconsistent — different analysts handle similar emails differently depending on their experience and how much time they have.

Your boss wants to explore whether an AI agent could handle the routine volume of this inbox, freeing human analysts to focus on complex cases that require judgment. She's asked you to build a prototype and evaluate whether it's trustworthy enough to pilot.

This is different from a chatbot or a dashboard. A chatbot answers questions it's asked. A dashboard shows pre-defined metrics. An agent *triages* — it reads an email it's never seen before, figures out what kind of issue it is, investigates using whatever tools are appropriate, and either handles it or escalates it. Each email may require a different workflow, and the agent figures out that workflow in real time. Your job is to build this, test it, and determine whether that value is real.

**Relevant Industry:** Media / Content Distribution Operations

**Data:**
- `./data/sakila_master.db` — MediaVault's operational database ([Sakila Sample Database, SQLite3](https://github.com/bradleygrant/sakila-sqlite3))
- `./data/emails.csv` — Customer email inbox
- `./data/mediavault_policies.md` — Business rules, escalation criteria, and communication guidelines
- `./data/schema_reference.md` — Human-readable database documentation

---

## Act 1: Understand the problem and context

### Step 0: Explore the data and scope the work in agents.md

Before designing your agent, you need to understand both the business data it will work with and the work it will be doing.

**The database.** MediaVault's operational database contains 16 interrelated tables covering films, customers, inventory, rentals, payments, staff, and stores. Load it and explore its structure.

In [ ]:
# write code below



**Check:**
- How many tables are in the database? What are they?
- Pick two or three tables and examine their columns and a few sample rows.
- Identify a pair of tables that are related by a foreign key. What does that relationship represent in business terms?
- Try writing a simple SQL query by hand (e.g., count of customers, or films by category). Does it return what you expect?

**The inbox.** Load the email corpus and read through a sample of 10–15 emails.

In [ ]:
# write code below



**Check:**
- What types of emails are in the inbox? Try to categorize them informally — billing questions, complaints, product inquiries, account requests, feedback, etc.
- For a few emails, trace what you would do to handle each one manually: What information would you need to look up? What policies would apply? What would your response look like? Which ones would you escalate, and to whom?
- Do the emails reference specific customers, films, or stores that exist in the database?

**The policies.** Read through `mediavault_policies.md`. Pay attention to the escalation criteria, the VIP and at-risk customer definitions, and the customer data handling rules.

**🍎 Food for thought:** Think about the range of emails you just read. Could a single report template or dashboard handle all of them? What makes some emails straightforward and others complex? What would an automated system need to be capable of to handle the full range?

---

Before moving forward, create a file named `agents.md` in the project root directory (likely the same level of the directory in which this notebook lives). This file specifies the intended role of AI in this project and serves as reference context for GitHub Copilot as you work.

Your `agents.md` must include the following five sections:

##### 1. What we're building
A one-sentence "elevator pitch" describing the prototype and its primary output (e.g., "An automated content enrichment system that classifies, tags, summarizes, and generates FAQs for news articles using transformer-based NLP models.").

##### 2. How AI helps solve the business problem
2–4 bullet points explaining the specific value-add of the AI components. Focus on the transition from the business "pain point" to the AI "solution."

##### 3. Key file locations and data structure
List the paths that matter (e.g., `./data/`).

##### 4. High-level execution plan
A step-by-step outline of the build process. Feel free to ask Copilot for help (or take a peek at the steps in Act II below) for a sense of structuring the work.

##### 5. Code conventions and constraints
To ensure the prototype remains manageable, add 1-2 bullet points specifying that code be as simple and straightforward as possible, using standard libraries unless instructed otherwise.

---

## Act 2: Build the agentic system

In this act, you will incrementally build an AI agent that triages and responds to customer emails for MediaVault. You will start with a provided orchestration loop — the infrastructure that allows an agent to use tools — and then add capabilities one at a time, observing how the agent's behavior changes with each addition.

Use GitHub Copilot as a development assistant, following a disciplined loop at every step:

- **Plan**: Have Copilot draft a clear, plain-language plan describing what needs to happen and in what order.
- **Validate**: Review and refine that plan to ensure it does exactly what the step requires — no more, no less.
- **Execute**: Have Copilot implement the validated plan in code.
- **Check**: Perform one or two concrete actions that confirm the code worked and that you understand the result.

**A note on workload:** Copilot should be doing the heavy lifting on implementation throughout this assignment. A tool like the SQL execution function in Step 2 — with read-only enforcement, error handling, and row limits — is something Copilot can produce in a single prompt. Your job is not to write every line of Python yourself. Your job is to *direct* Copilot (what to build and why), *understand* what it produces (can you explain how the guardrail works?), and *make design decisions* (one tool or two? what error format? what row limit?). The thinking is yours; the typing is Copilot's.

---

### Step 1: Understand the orchestration loop

An agent is not a single API call. It is a *loop*: the model receives a message, decides whether to call a tool, the system executes that tool, feeds the result back to the model, and the model decides what to do next. This continues until the model produces a final response with no further tool calls. The infrastructure that manages this cycle is called the **orchestration loop**.

Below is a minimal orchestration loop. Read through it carefully before running it. This is the foundation you will build on for the rest of the assignment.

```python
# ============================================================
# Orchestration Loop Starter Code
# ============================================================
# This code provides the basic infrastructure for an AI agent:
# - Connects to a language model
# - Sends messages with a list of available tools
# - Detects when the model wants to call a tool
# - Executes the tool and feeds the result back
# - Repeats until the model produces a final text response
#
# You will extend this throughout the assignment by adding
# new tools, modifying the system prompt, and adding checks
# to the loop itself.
# ============================================================

import json
from datetime import date

# --------------------------------------------------
# Model connection
# --------------------------------------------------
# Using github-copilot-sdk. Replace with your preferred
# provider (Anthropic, OpenAI, etc.) if needed.
# Consult Copilot for setup instructions for your provider.

# TODO: Set up your model client here.
# client = ...

# --------------------------------------------------
# Tool registry
# --------------------------------------------------
# Each tool has:
#   - A schema (name, description, parameters) that tells the
#     model what the tool does and how to call it
#   - A Python function that actually executes the tool
#
# The model sees only the schema. Your code calls the function.

def get_current_date():
    """Return today's date as a string."""
    return str(date.today())

# Tool schemas describe tools to the model.
# This format may vary by provider — adjust as needed.
tool_schemas = [
    {
        "name": "get_current_date",
        "description": "Returns today's date. Use when the user asks about the current date.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]

# Map tool names to their Python functions.
tool_functions = {
    "get_current_date": get_current_date
}

# --------------------------------------------------
# System prompt
# --------------------------------------------------
system_prompt = "You are a helpful assistant. You have access to tools and should use them when relevant."

# --------------------------------------------------
# The orchestration loop
# --------------------------------------------------
def run_agent(user_message, max_iterations=10):
    """
    Send a message to the agent and get a response.
    The agent may call tools multiple times before responding.
    """
    # Build the initial conversation
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]

    for i in range(max_iterations):
        # Call the model
        # TODO: Replace with your provider's API call.
        # response = client.chat(messages=messages, tools=tool_schemas)
        response = None  # placeholder

        # Check if the model wants to call a tool
        # TODO: Adapt this to your provider's response format.
        # The key logic: if the response contains a tool call,
        # execute it and continue. If not, return the text.

        if has_tool_call(response):
            tool_name, tool_args = extract_tool_call(response)
            print(f"  [Tool call] {tool_name}({tool_args})")

            # Execute the tool
            if tool_name in tool_functions:
                result = tool_functions[tool_name](**tool_args)
            else:
                result = f"Error: Unknown tool '{tool_name}'"

            print(f"  [Tool result] {result}")

            # Append the tool call and result to the conversation
            # TODO: Format these messages for your provider.
            # messages.append({"role": "assistant", "tool_call": ...})
            # messages.append({"role": "tool", "content": str(result)})

        else:
            # No tool call — return the model's text response
            final_text = extract_text(response)
            print(f"  [Final response] {final_text}")
            return final_text

    return "Error: Agent reached maximum iterations without producing a final response."


# --------------------------------------------------
# Helper functions
# --------------------------------------------------
# TODO: Implement these for your provider's response format.

def has_tool_call(response):
    """Check if the model's response contains a tool call."""
    pass

def extract_tool_call(response):
    """Extract the tool name and arguments from a tool call response."""
    pass

def extract_text(response):
    """Extract the final text content from a non-tool-call response."""
    pass
```

**Your task for this step:** Get this loop working with your chosen model provider. The `TODO` comments mark the places where you need to fill in provider-specific code. Use Copilot to help — ask it how to adapt the loop for your specific provider's tool-calling API.

**Plan**: Have Copilot help you complete the `TODO` sections for your chosen model provider.

**Validate**: The completed code should connect to your model, send messages, detect tool calls, execute them, and continue the loop.

**Execute**:

In [ ]:
# write code below



**Check:**
- Run `run_agent("What is today's date?")` — does it call the `get_current_date` tool and respond with the correct date?
- Run `run_agent("What is the capital of France?")` — does it answer directly *without* calling the tool?
- Look at the printed output. Can you see each step of the loop: the tool call, the result, and the final response?
- What happens if you ask something that might or might not need the tool, like "Is today a holiday?"?

**🍎 Food for thought:** The model doesn't "wait" for the tool result the way a program waits for a function to return. Look at the code — what's actually happening between the tool call and the result? Who is managing that handoff, the model or your code?

---

### Step 2: Give the agent access to the business data

Your agent will need to investigate customer issues, which means it needs access to MediaVault's operational database. Now you'll build your first real tool and register it with the orchestration loop.

**Plan**: Build a database tool that allows the agent to:
- Discover what tables and columns exist in the database
- Execute SQL queries and receive results
- Handle errors gracefully — if a query fails, the error message should come back to the agent so it can try a different approach

The tool must enforce guardrails:
- **Read-only**: Only `SELECT` statements should be permitted. Any query containing `INSERT`, `UPDATE`, `DELETE`, `DROP`, `ALTER`, or `CREATE` should be rejected before execution.
- **Row limits**: Cap returned results (e.g., 100 rows) to avoid overwhelming the model's context.

Whether you implement this as one tool or multiple is your design decision.

**Validate**: Review the plan. Does it handle schema discovery? Read-only enforcement? Error messages the model can learn from? Result size limits?

**Execute**: Build the tool(s), add the schema(s) to `tool_schemas`, add the function(s) to `tool_functions`, and update the system prompt to describe the agent's new capability.

In [ ]:
# write code below



**Check:**
- Ask "How many films are in our catalog?" — does the agent write and execute correct SQL?
- Ask "What are our top 5 film categories by number of titles?" — does it handle aggregation and ordering?
- Ask "Which customers have spent the most money?" — does it join the right tables?
- Ask "Delete all records from the payment table" — does the guardrail catch this?
- Ask about a nonexistent table — does the agent get the error message and recover?

**🍎 Food for thought:** Does the agent explore the schema before writing SQL, or does it guess from its training knowledge? What happens when it guesses wrong — does recovery look like planning or trial-and-error?

---

### Step 3: Give the agent business knowledge

Data alone doesn't answer business questions. When a customer emails asking why they were charged a late fee, the right response depends on company policy — not just what's in the database. Your agent needs access to business rules.

**Plan**: Build a tool that gives the agent access to MediaVault's policy document (`./data/mediavault_policies.md`). Consider your design options: Should the agent receive the full document every time? Search it by keyword? Query it by policy area? There are tradeoffs — choose an approach and be prepared to justify it in your README.

**Validate**: The tool should give the agent enough context to apply business rules without flooding its context window with irrelevant information.

**Execute**: Build the tool and register it with the orchestration loop.

In [ ]:
# write code below



**Check:**
- Ask "Is customer #42 eligible for VIP status?" — does the agent consult the policy document to learn the VIP criteria, *then* query the database to check the customer's history?
- Ask "What's our policy on overdue rentals?" — does it retrieve the right information?
- Ask "Is Store 1 performing well?" — does it reference the performance evaluation criteria from the policies, then gather relevant data?
- Ask a question where the policy is ambiguous or doesn't clearly apply. How does the agent handle uncertainty?

In [ ]:
# write code below


**🍎 Food for thought:** This tool is doing something similar to the RAG system you built in Week 7 — retrieving context to ground the model's response. But here it's one capability among several, invoked at the agent's discretion. When does the agent choose to consult the policies versus just answering from data? Is it making that choice well?

### Step 4: Give the agent the ability to draft responses

Your agent can now investigate issues. But handling an email means *responding* to it — and a customer-facing response is a fundamentally different kind of output than an internal data lookup.

**Plan**: Build a tool that drafts a customer email response. The tool should take context — the original email, investigation findings, relevant policies — and produce a formatted reply following MediaVault's communication guidelines (see the Communication Guidelines section of the policies document).

Think about the design: Does this tool just prompt the model with a template? Does it enforce structural requirements (greeting, reference number, clear next steps, sign-off)? Does it validate the draft against any rules (no internal jargon, no other customers' data, no promises the company can't keep)?

**Validate**: A drafted response should be professional, accurate to the investigation findings, policy-compliant, and appropriate for a real customer to receive.

**Execute**: Build the tool and register it with the orchestration loop.

In [ ]:
# write code below



**Check:**
- Manually pass the agent a billing complaint scenario: "A customer says they were charged twice. You've confirmed there's a duplicate payment of $4.99." Does the draft acknowledge the issue, reference the finding, and describe next steps?
- Pass a scenario where the customer is wrong: "A customer says they were overcharged, but the charges match the rental rate and late fee policy." Does the draft maintain a professional tone while explaining the charges?
- Pass a scenario involving another customer's data — does the draft avoid leaking it?

In [ ]:
# write code below


**🍎 Food for thought:** The same model is now the "brain" deciding what to investigate AND the "pen" drafting the response. Could these be different models? Why might you want that in production? What does it mean to trust a model to write something a customer will read?

---

### Step 5: Give the agent the ability to escalate

Not every email should be handled automatically. Some require human judgment — because they're sensitive, because they fall outside the agent's authority, or because the agent is uncertain. Your agent needs the ability to recognize these situations and hand off gracefully.

**Plan**: Build an escalation tool that flags an email for human review. The tool should produce a structured escalation ticket:
- The original email
- A summary of what the agent investigated (if anything)
- The reason for escalation
- A suggested routing (billing team, store manager, legal, etc.)

The escalation criteria in `mediavault_policies.md` describe what should be escalated. But the agent will need to *recognize* these situations from the email content — there's no pre-classification telling it what type each email is.

**Validate**: The escalation ticket should contain enough context for a human to pick up where the agent left off without re-reading the entire conversation history.

**Execute**: Build the tool and register it with the orchestration loop.

In [ ]:
# write code below



**Check:**
- Feed the agent an email from a customer threatening legal action — does it escalate? Is the routing suggestion reasonable?
- Feed it an email asking to delete account data — does it escalate?
- Feed it a routine billing question — does it handle it normally rather than unnecessarily escalating?
- Feed it a genuinely ambiguous email — one that could go either way. What does the agent do? Is that the right call?

In [ ]:
# write code below

**🍎 Food for thought:** The agent isn't calling a classification tool to decide whether to escalate — it's making a judgment call in its reasoning. Is that reliable enough? When would you want something more deterministic — a rules-based classifier, a keyword filter — instead of relying on the model's judgment? What are the costs of escalating too much versus too little?

---

### Step 6: Add an approval gate

You now have an agent that can investigate, respond, and escalate. But should its outputs go straight to customers — or to the database — without any structural check?

An approval gate is a checkpoint in the orchestration loop that reviews the agent's actions *before* they execute. It operates on what the agent is *doing*, not what it's *thinking* — it's a safeguard independent of the model's own judgment.

**Plan**: Modify your orchestration loop to add a check *before* each tool call is executed and *before* any drafted response is finalized. Consider what the gate should catch:
- SQL queries that access PII columns (email, address, phone)
- Drafted responses that contain internal data the customer shouldn't see
- Emails that match escalation criteria but that the agent is trying to handle directly
- Any data modification attempts that slipped past the SQL tool's own guardrails

Implement this in two ways and compare them:
1. **Infrastructure-level**: Python logic in the orchestration loop that inspects the tool call or drafted response before execution
2. **Prompt-level**: Instructions in the system prompt directing the model to review its own work before submitting

**Validate**: The gate should be able to approve, flag (with explanation), or block actions.

**Execute**: Modify your `run_agent` function to include the gate, and implement both approaches.

In [ ]:
# write code below



**Check:**
- Ask the agent to "Look up the email addresses for all customers who are overdue" — does the gate flag the PII access?
- Ask "What's the average rental revenue per store?" — does this pass through without friction?
- Feed it an email about a staff complaint — if the agent tries to handle it directly rather than escalating, does the gate catch it?
- Run the same test cases through both your infrastructure-level and prompt-level approaches. Which catches more problems? Which produces false positives? Which would you trust in production?

In [ ]:
# write code below



**🍎 Food for thought:** You now have trust boundaries at multiple layers — guardrails in the SQL tool, escalation logic in the agent's reasoning, and an approval gate in the orchestration loop. Is this redundant, or is redundancy the point? There's a fundamental design tension here: do you trust the agent to police itself (via the system prompt), or do you enforce constraints in the infrastructure? What are the tradeoffs of each approach?

---

### Step 7: Design a skill — Email Triage

You've built individual tools: data access, policy knowledge, response drafting, escalation, and an approval gate. Each one is atomic — it does one thing when called. But handling a customer email requires *composing* these tools into a coherent workflow, where the agent's choices at each stage depend on what it learned in the previous stage. That workflow is a **skill**.

A tool is like a function. A [skill](https://code.claude.com/docs/en/skills) is like a playbook. It defines a strategy for a type of task — what to do first, what to check, where the decision points are, and how different situations lead to different paths through the tools.

**Plan**: Before you run your agent against real emails, design the skill explicitly. Think back to the emails you read in Step 0 and the manual triage process you described in `agents.md`. Now that you've built the tools, you can be specific: what does the agent do first when it receives an email? What does it check? Where does the workflow branch depending on what it finds?

Create a `SKILL.md` file that describes the Email Triage skill. It should include:

1. **Trigger**: What initiates this skill? (An incoming customer email.)

2. **Workflow stages**: Walk through the stages of handling an email, in order. For each stage, describe:
   - What the agent should do
   - Which tool(s) it would use
   - What information it needs from previous stages
   - What decisions it makes before proceeding to the next stage

3. **Branching logic**: Where does the workflow diverge based on what the agent finds? A billing dispute follows a different path than a product inquiry, which follows a different path than an angry customer threatening to cancel. Map out at least three distinct paths and describe how the agent should recognize which path to take.

4. **Stopping conditions**: How does the agent know it's done? What constitutes a completed email — a drafted response that passes the approval gate? An escalation ticket? What happens if the agent gets stuck or can't resolve the issue?

5. **What the skill does NOT cover**: What types of emails or situations fall outside this skill? What would require a different skill entirely, or a human from the start?

**Validate**: Your skill design should account for at least three distinct types of emails that follow meaningfully different paths through the tools. If every email follows the same sequence regardless of content, the skill isn't capturing the adaptive behavior that makes an agent valuable.

**Execute**: 

In [ ]:
# write code below

**Check**: Pick three emails from the corpus — one straightforward, one complex, one that should be escalated. Trace each one through your `SKILL.md` on paper. Does the workflow handle each one sensibly? Does the branching logic correctly route each email to a different path? Are there any points where the workflow is ambiguous about what the agent should do next?

**🍎 Food for thought:** Look at your `SKILL.md`. How much of this strategy will live in the system prompt versus in the code of your orchestration loop? Could you hand this document to someone with the same tools but a different system prompt and have them reproduce your agent's behavior? What does that tell you about where the "intelligence" of an agentic system actually resides — in the model, in the tools, or in the skill design?

---

### Step 8: Process the inbox

This is the payoff. You have an agent with tools for data access, policy knowledge, response drafting, and escalation — with an approval gate checking its work. Now you'll run it against a batch of real customer emails and observe what happens.

**Plan**: Set up a batch process that:
- Reads a sample of 10 emails from the corpus (select a diverse set — make sure you have a mix of billing questions, complaints, product inquiries, account requests, and at least one or two that should be escalated)
- Feeds each email to your agent
- Captures a complete trace for each: the original email, every tool call and result, every reasoning step, any gate decisions, and the final output (drafted response or escalation ticket)

Implement logging that captures this full trace. For any email, you should be able to reconstruct the complete chain of the agent's decisions.

**Validate**: Each trace should show the full journey from input email to final output, with every intermediate step visible.

**Execute**:

In [ ]:
# write code below



**Check**: For 4–5 of the emails — choose ones that represent different types and complexity levels — do a deep annotation of the trace (in comments, in a separate markdown file, or inline):
- What type of email was this? How did the agent classify it — explicitly or implicitly?
- Did the agent use the right tools in a sensible order?
- Did it consult business policies when relevant?
- Did the approval gate trigger when it should have? Did it trigger when it shouldn't have?
- If the agent drafted a response: was it accurate, professional, and policy-compliant?
- If the agent escalated: was escalation warranted? Was the ticket useful for a human reviewer?
- Where, if anywhere, did the agent's reasoning appear shallow, circular, or confused?

For the remaining emails, note the key patterns briefly:
- How many emails did the agent handle autonomously vs. escalate?
- What was the average number of tool calls per email? What was the range?
- Were there emails the agent handled confidently but got wrong?
- Were there emails it escalated unnecessarily?
- Can you see the agent adapting its approach based on email type, or does it follow the same pattern regardless of what it reads?

In [ ]:
# write code below


**🍎 Food for thought:** Pick one email the agent handled well and one it handled poorly. Look at the traces side by side. Can you tell from the reasoning alone — before seeing the final output — which would succeed and which would fail? What does that suggest about the nature of the model's "reasoning"? Is it planning, or is it pattern-matching that sometimes lands and sometimes doesn't?

---

---

## End of Act II

At this point, you should have direct evidence of how an AI agent might handle your workflow and what quality/performance tradeoffs exist. Use these observations to inform Act III discussions with stakeholders.

Before moving on to Act III, create a file named `README.md` in the project root.

This README should capture the current state of the prototype as if you were handing it off to a colleague. Keep it concise and grounded in what actually exists.

### 1. What this prototype does
In one sentence, clearly describe the capability that was built and the problem it is intended to address.

### 2. How it works (at a high level)
In a few bullet points, specify:
- what data the system operates over,
- what models, representations, and tools it uses,
- how results are produced.

### 3. Limitations and open questions
Briefly note:
- the most important limitations you observed or conceive of, and
- any open questions that would need to be addressed before broader use.

## Act 3: Socialize the solution with stakeholders

You've built a prototype agent for customer email triage. Now imagine presenting it to three colleagues at MediaVault who would be affected by deploying this technology. Use GitHub Copilot in **Ask** mode to roleplay as each colleague.

---

### Stakeholder 1: Customer Service Team Lead

This person currently manages the team that handles this inbox manually. They've been doing this for years and take pride in the quality of their team's responses. They're intrigued by the idea of automation but protective of the customer relationship.

**Prompt Copilot to respond as this stakeholder:**
- What would make them trust the agent's responses? What would immediately erode that trust?
- How would they want to handle the inevitable situation where the agent sends a customer a wrong or tone-deaf response?
- They've seen their newer analysts make mistakes too — how is this different?

---

### Stakeholder 2: Head of IT and Data Infrastructure

This colleague manages MediaVault's data systems and is responsible for security, access controls, and system reliability. They've sat through AI demos before and want to understand what's actually happening under the hood.

**Prompt Copilot to respond as this stakeholder:**
- What concerns would they have about an AI agent executing SQL queries against the operational database?
- What would they think of your approval gate? What would they want to add to it?
- How would they want to audit and monitor the agent's activity in production? What logging would they require?

---

### Stakeholder 3: VP of Operations

This colleague oversees all store operations and reports to the CEO. They care about efficiency, consistency across locations, and making sure technology investments have clear ROI.

**Prompt Copilot to respond as this stakeholder:**
- What percentage of inbox volume would this need to handle to justify the investment? How would they think about the cost per email versus a human analyst?
- How would they measure whether customers are getting *better* service from the agent vs. the current process?
- Who "owns" the agent? Who decides what it can and can't do? Who is accountable when it's wrong?

---

**Document each conversation** by saving or exporting it from GitHub Copilot. These perspectives will help you anticipate real-world concerns when presenting AI solutions to business stakeholders.

---

## Submission

1. **Save your files**: Ensure your code files, `agents.md`, `README.md`, and trace annotations are all saved.
2. **Export your chat history**: Save or export your GitHub Copilot chat conversations, including the stakeholder roleplay sessions.
3. **Upload to Canvas**: Submit all files to [Canvas](https://canvas.northwestern.edu/courses/245397/assignments/1668987).